In [2]:
# # Install required packages
!pip install PyPDF2 google-api-python-client google-auth-httplib2 google-auth-oauthlib xlsxwriter

import os
import io
import pandas as pd
import PyPDF2
from google.colab import drive, auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.auth import default

# === CONFIGURATION ===
FOLDER_PATH = "My/ORDERS/HIGH COURTs"  # <--- Your folder path
OUTPUT_DIR = '/content/drive/My Drive/colab excel'  # Modified path
EXCEL_FILENAME = 'Legal_Documents_Analysis.xlsx'
KEYWORDS = [
    "dismissed",
    "disposed of",
    "no merit",
    "stay",
    "issue notice",
    "list it",
    "delay condoned",
    "leave granted"
]

# === SETUP ===
# Mount Google Drive
drive.mount('/content/drive')

# Authenticate for API access
auth.authenticate_user()
creds, _ = default()

# Build the Drive API service
service = build('drive', 'v3', credentials=creds)

# Ensure output directory exists (modified creation)
os.makedirs(OUTPUT_DIR, exist_ok=True)  # Better directory creation

def find_folder_id(service, folder_path):
    """Find the folder ID for a given path like 'My/ORDERS/HIGH COURTs'"""
    parent_id = 'root'
    for folder_name in folder_path.split('/'):
        folder_name = folder_name.strip()
        query = (
            f"name = '{folder_name}' and '{parent_id}' in parents and "
            "mimeType = 'application/vnd.google-apps.folder'"
        )
        results = service.files().list(q=query, fields="files(id, name)").execute()
        folders = results.get('files', [])
        if not folders:
            raise ValueError(f"Folder not found: '{folder_name}' in path '{folder_path}'")
        parent_id = folders[0]['id']
    return parent_id

def get_pdf_files(service, folder_id):
    """Get all PDFs in the specified folder"""
    query = f"'{folder_id}' in parents and mimeType='application/pdf'"
    results = service.files().list(
        q=query,
        fields="files(id, name, webViewLink, size)",
        pageSize=1000
    ).execute()
    return results.get('files', [])

def check_pdf_images(pdf_reader):
    """
    Check if PDF contains images by examining each page.
    Returns True if any images are found, False otherwise.
    """
    try:
        has_images = False
        image_count = 0

        # Check each page for images
        for page_num, page in enumerate(pdf_reader.pages):
            try:
                # Check if page has images using the images property
                if hasattr(page, 'images') and page.images:
                    page_image_count = len(page.images)
                    if page_image_count > 0:
                        has_images = True
                        image_count += page_image_count
                        print(f"  📷 Found {page_image_count} image(s) on page {page_num + 1}")
            except Exception as e:
                # Some PDFs might have corrupted image data, continue checking other pages
                print(f"  ⚠️ Error checking images on page {page_num + 1}: {str(e)}")
                continue

        return has_images, image_count

    except Exception as e:
        print(f"  ❌ Error during image detection: {str(e)}")
        return False, 0

def analyze_pdf(service, file_id):
    """Extract text, page count, and image information from a PDF file, handle errors."""
    try:
        print(f"  📄 Downloading and analyzing PDF...")
        request = service.files().get_media(fileId=file_id)
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()

        fh.seek(0)
        pdf = PyPDF2.PdfReader(fh)

        # Extract text
        text = "\n".join(page.extract_text() or '' for page in pdf.pages).lower()

        # Check for images
        has_images, image_count = check_pdf_images(pdf)

        return {
            'text': text,
            'pages': len(pdf.pages),
            'has_images': has_images,
            'image_count': image_count,
            'error': None
        }
    except Exception as e:
        return {
            'text': '',
            'pages': 0,
            'has_images': False,
            'image_count': 0,
            'error': str(e)
        }

def process_files():
    """Main processing function"""
    try:
        print(f"Locating folder: {FOLDER_PATH}")
        folder_id = find_folder_id(service, FOLDER_PATH)
        pdf_files = get_pdf_files(service, folder_id)

        if not pdf_files:
            print("No PDF files found in the specified folder.")
            return

        print(f"Found {len(pdf_files)} PDF files. Starting analysis...\n")

        results = []

        for i, file in enumerate(pdf_files, 1):
            print(f"📁 Processing file {i}/{len(pdf_files)}: {file['name']}")

            file_info = {
                'File Name': file['name'],
                'Drive Link': file['webViewLink'],
                'Categories': [],
                'Pages': 0,
                'Has Images': 'No',
                'Image Count': 0,
                'Status': 'OK'
            }

            analysis = analyze_pdf(service, file['id'])

            if analysis['error']:
                file_info.update({
                    'Categories': ['Corrupted'],
                    'Status': analysis['error']
                })
                print(f"  ❌ Error processing file: {analysis['error']}")
            else:
                file_info['Pages'] = analysis['pages']
                file_info['Has Images'] = 'Yes' if analysis['has_images'] else 'No'
                file_info['Image Count'] = analysis['image_count']

                text = analysis['text']

                # Short document category
                if 1 <= file_info['Pages'] <= 5:
                    file_info['Categories'].append('Short Document')

                # Image category - NEW FEATURE
                if analysis['has_images']:
                    file_info['Categories'].append('image')
                    print(f"  ✅ Contains {analysis['image_count']} image(s)")

                # Keyword matching
                found_keywords = [kw for kw in KEYWORDS if kw in text]
                if found_keywords:
                    file_info['Categories'].extend(found_keywords)
                    print(f"  🔍 Found keywords: {', '.join(found_keywords)}")

                # Special category for "no merit"
                if 'no merit' in text and 'no merit' not in file_info['Categories']:
                    file_info['Categories'].append('no merit')

                if not file_info['Categories']:
                    file_info['Categories'].append('No Match')

                print(f"  📊 Pages: {file_info['Pages']}, Categories: {', '.join(file_info['Categories'])}")

            results.append(file_info)
            print()  # Add spacing between files

        # Save to Excel
        excel_path = os.path.join(OUTPUT_DIR, EXCEL_FILENAME)
        print(f"💾 Saving results to Excel: {excel_path}")

        with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
            # Prepare dataframe
            df_results = []
            for result in results:
                df_results.append({
                    'File Name': result['File Name'],
                    'Drive Link': result['Drive Link'],
                    'Categories': ', '.join(result['Categories']) if isinstance(result['Categories'], list) else result['Categories'],
                    'Pages': result['Pages'],
                    'Has Images': result['Has Images'],
                    'Image Count': result['Image Count'],
                    'Status': result['Status']
                })

            df = pd.DataFrame(df_results)

            # Main sheet with all data
            df.to_excel(writer, sheet_name='All Documents', index=False)

            # Category sheets (updated to include images)
            categories = {
                'Documents with Images': 'image',  # NEW CATEGORY
                'Dismissed Cases': 'dismissed',
                'Disposed Cases': 'disposed of',
                'Stay Orders': 'stay',
                'Notices Issued': 'issue notice',
                'Listing Orders': 'list it',
                'No Merit Cases': 'no merit',
                'Short Documents': 'Short Document',
                'Corrupted Files': 'Corrupted'
            }

            for sheet_name, category in categories.items():
                if category == 'image':
                    # For images, filter by 'Has Images' column
                    cat_df = df[df['Has Images'] == 'Yes']
                else:
                    # For other categories, check in Categories column
                    cat_df = df[df['Categories'].str.contains(category, case=False, na=False)]

                if not cat_df.empty:
                    cat_df.to_excel(writer, sheet_name=sheet_name, index=False)
                    print(f"  📋 Created sheet '{sheet_name}' with {len(cat_df)} files")

        # Print summary
        print(f"\n✅ Analysis complete! Results summary:")
        print(f"   📁 Total files processed: {len(results)}")
        print(f"   📷 Files with images: {len([r for r in results if r['Has Images'] == 'Yes'])}")
        print(f"   🔄 Files with errors: {len([r for r in results if 'error' in r.get('Status', '')])}")
        print(f"   💾 Excel file saved to: {excel_path}")
        print(f"\n📂 Check your Google Drive: My Drive > colab excel > {EXCEL_FILENAME}")

    except Exception as e:
        print(f"❌ Error: {str(e)}")
        print("Troubleshooting Tips:")
        print("1. Verify folder path exists exactly as: 'My/ORDERS/HIGH COURTs'")
        print("2. Check folder names match exactly (including spaces and capitalization)")
        print("3. Confirm you have view access to the folder")

# Run the analysis
process_files()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Locating folder: My/ORDERS/HIGH COURTs


Found 16 PDF files. Starting analysis...

📁 Processing file 1/16: 2024-08-14_In the matter of Glas Trust Company LLC vs. Byju R (1).pdf
  📄 Downloading and analyzing PDF...
  🔍 Found keywords: stay, issue notice
  📊 Pages: 3, Categories: Short Document, stay, issue notice

📁 Processing file 2/16: 2023-12-15_In the matter of Maneesh Pharmaceuticals Ltd. vs.E.pdf
  📄 Downloading and analyzing PDF...
  🔍 Found keywords: dismissed, disposed of, stay
  📊 Pages: 5, Categories: Short Document, dismissed, disposed of, stay

📁 Processing file 3/16: 2020-07-20_In the matter of Saurabh Jain and Anr. Vs. Union o.pdf
  📄 Downloading and analyzing PDF...
  📊 Pages: 2, Categories: Short Document

📁 Processing file 4/16: 2018-01-10_In the matter of SAL Steel Limited Vs. Besto Trade.pdf
  📄 Downloading and analyzing PDF...
  📷 Found 1 image(s) on page 1
  📷 Found 1 image(s) on page 2
  📷 Found 1 image(s) on page 3
  📷 Found 1 image(s) on page 4
  ✅ Contains 4 image(s)
  📊 Pages: 4, Categories: Short Do